<style>
.sam-table{font-size:13px!important;line-height:1.5!important;width:100%!important;table-layout:auto!important}
.sam-table th,.sam-table td{padding:10px 12px!important;word-break:normal!important;overflow-wrap:normal!important}
.sam-table td{white-space:nowrap!important}
.sam-table td:first-child,.sam-table th:first-child{min-width:165px;white-space:normal!important}
.sam-wide-table{min-width:1120px!important}
.sam-wide-table th{min-width:90px;white-space:normal!important}
.sam-scroll{width:100%;overflow-x:auto;overscroll-behavior-x:contain;margin:12px 0 28px}
.sam-scroll .sam-table{margin-bottom:0!important}
.sam-table caption{padding-bottom:12px!important}
.sam-title{font-weight:650!important;line-height:1.22!important}
</style><div class="sam-kicker">Princeton · Rachel / FairVote data · November 13, 2026</div>
<div class="sam-title">Alaska 2026: crossover preferences,<br>vote transfers, and a single candidate axis</div>

**Prepared for Sam · Rachel’s November 11 ballot files**

This notebook answers the ballot questions and repeats the original one-axis Senate analysis using Rachel’s three CSVs only. **Daniel J. Sullivan Jr. is the outsider; Dan S. Sullivan is the incumbent senator.** The source covers 244,705 scanned Senate ballots and is preliminary.

<div class="sam-cards"><div class="sam-card"><div class="n">568</div><div class="l">Peltola above Dan S. · both legally usable</div></div><div class="sam-card"><div class="n">765</div><div class="l">Only Peltola in usable finalist rankings</div></div><div class="sam-card"><div class="n">61.5%</div><div class="l">Observed consistency · 3–4 named ranks</div></div><div class="sam-card"><div class="n">41.6%</div><div class="l">Random consistency · 49 fresh profiles</div></div></div>

### Findings to take into the meeting

- **568 of 3,940 literal Daniel-first ballots (14.42%) have both finalists in their usable rankings, with Peltola above Dan S.** The reverse count is **423 (10.74%)**. Headline comparisons now apply Alaska’s overvote and skipped-rank rules consistently.
- The governor/House distributions and every Senate Sankey count are unchanged from the previous analysis.
- **One-axis consistency is 61.54% observed versus 41.57% random** among three-/four-name prefixes, a 19.97-point difference.
- All 54 published round-count values match. The supplementary visible-name audit matches nine subgroup cells in the archived November 12 article using November 11 data; these are distinct from the headline countable-preference results.

<div class="sam-note"><b>First choice:</b> the descriptive questions use literal rank 1. All headline preference comparisons, the Sankey and MDS use usable preferences after skipped-rank and overvote rules. We retain the same literal-rank-1 cohort so this change does not silently change the denominator. Candidate affiliations are not voters’ registered parties.</div>

In [ ]:
from pathlib import Path
import sys,json,csv,gzip,hashlib,contextlib,io,os
import numpy as np
import pandas as pd
from IPython.display import display,HTML,Image
ROOT=Path.cwd().resolve()
assert (ROOT/'inputs/Alaska_20261103_US_Senate_11_11.csv').is_file(), 'Open from the Rachel package folder.'
WORKSPACE=ROOT.parent
RACHEL_INPUT_READS=set()
# Enforce the source policy: workspace data outside this Rachel package cannot be opened.
def rachel_source_guard(event,args):
    if event!='open' or not args or not isinstance(args[0],(str,bytes,os.PathLike)):return
    try:p=Path(os.fsdecode(args[0])).resolve()
    except (ValueError,OSError):return
    if WORKSPACE in p.parents and ROOT not in p.parents and p.suffix.lower() in {'.csv','.tsv','.gz','.zip','.json','.sqlite','.xlsx'}:
        raise PermissionError('Rachel-only analysis: external workspace data source blocked: '+str(p))
    mode=args[1] if len(args)>1 else None
    if ROOT/'inputs' in p.parents and p.suffix.lower()=='.csv' and (not isinstance(mode,str) or 'r' in mode):
        RACHEL_INPUT_READS.add(p.name)
sys.addaudithook(rachel_source_guard)
for folder in [ROOT,ROOT/'descriptive',ROOT/'mds',ROOT/'flows']:sys.path.insert(0,str(folder))
import presentation as show
show.theme()
print('Source policy active: only the supplied Rachel ballot CSVs are used.')


## 1 · Rebuild and link Rachel’s files

The source files contain one row per ballot/contest, five literal rank tokens, ballot identifiers, geography/counting metadata, and first-rank ambiguity/overvote annotations. Every original 15-column row is preserved. A named cell, `skipped`, and `overvote` remain distinct.

IDs are normalized only for joining: trailing underscores and zero-padding are removed from the three numeric ID components. We check for collisions and join by ID, never row position. **No older source fills absent records or hidden overvote identities.**

In [ ]:
import descriptive_analysis
with contextlib.redirect_stdout(io.StringIO()):
    descriptive_analysis.ingest(ROOT/'inputs')
    descriptive_analysis.analyze()
    import verify_rachel
    descriptive_verification=verify_rachel.main()
    import verify_usable_preferences
    usable_verification=verify_usable_preferences.main()
descriptive=json.loads((ROOT/'descriptive/descriptive_results.json').read_text())
provenance=json.loads((ROOT/'descriptive/source_provenance.json').read_text())
assert provenance['id_join']['senate_house_ids_identical']
assert provenance['id_join']['governor_subset_of_federal']
assert provenance['id_join']['missing_governor_rows']==130
show.table(pd.DataFrame([{'Source':slug,'Ballots':f"{provenance['rows'][slug]:,}",'File':meta['filename']} for slug,meta in provenance['inputs'].items()]),'Rachel-only source coverage')
show.table(pd.DataFrame([{'Senate literal first choice':name,'Ballots':f'{count:,}'} for name,count in descriptive['literal_rank1_senate_cohorts'].items()]),'All Senate first-choice categories; no ballots dropped')


### Data quality and original rankings

The full audit independently reconstructs every original source row, normalized ID, countable prefix and cross-office table. The original CSVs and all 15 supplied columns remain unchanged. The checks also cover first-rank ambiguity annotations; those annotations are retained without overriding Rachel’s rank cells or pretending to re-adjudicate ballot images.

The flags below overlap. “One named cell” means exactly one visibly named cell and no overvote anywhere, including write-in as a name. It differs from one usable preference after counting barriers. Raw named-cell lengths count repeat names and names below barriers; usable named lengths collapse repeats and exclude generic write-ins only after applying the barriers.

In [ ]:
import importlib.util
quality_spec=importlib.util.spec_from_file_location('rachel_quality_audit',ROOT/'audit_ultra/descriptive/independent_descriptive_audit.py')
quality_module=importlib.util.module_from_spec(quality_spec)
quality_spec.loader.exec_module(quality_module)
with contextlib.redirect_stdout(io.StringIO()):quality_module.main()
quality=json.loads((ROOT/'audit_ultra/descriptive/independent_descriptive_audit.json').read_text())
assert quality['status']=='passed'
quality_rows=[]
for race,label in [('governor','Governor'),('us_house','House'),('us_senate','Senate')]:
    q=quality['data_quality'][race]['counts']
    quality_rows.append({'Contest':label,'Rows':q['rows'],'One named cell, no overvote':q['exact_one_named_cell_no_overvote'],'Any overvote':q['any_overvote_token'],'Overvote at rank 1':q['literal_rank1_overvote'],'Internal skip':q['any_internal_skip'],'Repeated name':q['any_repeat_named_candidate']})
show.table(pd.DataFrame(quality_rows),'Full-source quality counts; overlapping flags')
senate_quality=quality['data_quality']['us_senate']
show.table(pd.DataFrame([{'Length':k,'Original named cells (includes write-in/repeats)':senate_quality['literal_named_cell_length_distribution'].get(str(k),0),'Usable distinct named choices (excludes write-in)':senate_quality['usable_prefix_length_excluding_writein_distribution'].get(str(k),0)} for k in range(6)]),'Senate rank lengths: original cells versus countable model input')


## 1 · Daniel J. Sullivan Jr. first-choice ballots

**Daniel J. Sullivan Jr. is the outsider; Dan S. Sullivan is the incumbent senator.** The cohort remains **3,940 ballots explicitly naming Daniel J. at literal rank 1**. Keeping this cohort fixed lets us isolate the effect of applying Alaska’s counting rules to later preferences.

For the headline comparison, retain only the countable prefix: **keep earlier valid preferences, stop before an overvote, cross one internal skip, and stop at two consecutive internal skips**. Repeated candidates count once. Original Rachel records stay intact; no whole ballot is discarded merely for containing a later overvote.

**568 ballots (14.42%) have both finalists usable and place Peltola above Dan S. Sullivan.** The reverse count is **423 (10.74%)**. Another **765** have only Peltola in their usable finalist rankings, **481** have only Dan S., and **1,703** have neither finalist available in the usable prefix.

These are exact counts under the stated counting interpretation. Candidate identities hidden inside an excluded overvote do not need to be reconstructed.

In [ ]:
show.preference_table()
usable=descriptive['question1_usable_prefix_categories']
show.table(pd.DataFrame([
 {'Final usable destination':'Peltola','Ballots':usable['peltola_above_senator_both_ranked']+usable['peltola_only_explicitly_ranked']},
 {'Final usable destination':'Dan S. Sullivan','Ballots':usable['senator_above_peltola_both_ranked']+usable['senator_only_explicitly_ranked']},
 {'Final usable destination':'Neither finalist / inactive','Ballots':usable['neither_explicitly_ranked']}
]),'Same 3,940 ballots: 1,333 Peltola, 904 Dan S., and 1,703 inactive')


**What changed:** names inside or below an overvote no longer contribute to the headline ranking comparison. Raw visible-name descriptions are preserved in the supplementary definition audit, where they are explicitly distinguished from countable preferences.

**Cohort sensitivity:** the separate first-structurally-usable-choice cohort contains **3,992** Daniel ballots. Under the same countable-prefix treatment, its both-finalist counts are **574 Peltola-above** and **430 Dan-above**. We do not substitute this larger cohort for the main literal-rank-1 comparison.

## 2 · Governor and House choices within each Senate first-choice group

Each cell gives **ballots (percentage of its full Senate cohort)**. Main tables use literal rank 1 in every race. Blanks, overvotes, write-ins and the 130 governor-file absences remain explicit categories. Those absent governor rows are not filled and are not treated as blank votes.

**Party means the selected candidate’s or ticket’s affiliation.** Rachel’s CSVs contain no voter-registration party column. We use an explicit static candidate-party annotation lookup: Kreiss-Tomkins/Johnson is Democrat/Nonpartisan; Bill Hill is Nonpartisan; Daniel J. has no affiliation printed on the sampled general-election ballot (the official candidate list says Undeclared). All ballot counts are derived only from Rachel’s files.

In [ ]:
show.cross_table('governor')
show.cross_table('us_house')
show.party_plot()
show.cross_table('governor',party=True)
show.cross_table('us_house',party=True)


### First-usable-choice sensitivity

These tables use the structural first usable choice consistently in all three contests, allowing one internal skipped rank and retaining a generic write-in as a named first usable option. They are labeled separately from literal-rank-1 tables.

In [ ]:
show.cross_table('governor',sensitivity=True)
show.cross_table('us_house',sensitivity=True)


## Supplementary audit · visible names versus countable preferences

This is a supplementary recorded-name description, not the main Alaska-rule result. Rachel’s visible rank columns reproduce the Daniel subgroup totals in the **archived FairVote article updated November 12, using the November 11 CVR**, exactly when we select the **first explicitly named finalist, ignoring counting barriers**:

- Peltola first among visible finalists: **1,345**.
- Dan S. first among visible finalists: **910**.
- Neither finalist explicitly named: **1,685**.

Under the documented overvote/skip barriers, the same literal-first-choice cohort instead yields **1,333 Peltola**, **904 Dan S.**, and **1,703 inactive**. That second calculation describes usable transfers. We do not label the first calculation an official-rule transfer tally. Reproducing the article’s descriptive numbers does not prove which internal code FairVote used.

The article labels its residual governor group “blank.” Our matching **broader residual category** has **126 blank first ranks, 39 write-ins, 3 overvotes, and 1 absent governor row**. We preserve that distinction. The archived article and retrieval record are included; its subsequently updated live version uses newer data and is not the benchmark here.

In [ ]:
flags=descriptive['question1_flags_and_finalist_transfers'];visible=descriptive['question1_visible_named_slot_categories']
show.table(pd.DataFrame([
 {'Destination':'Peltola','First visible named finalist':visible.get('peltola_above_senator_both_ranked',0)+visible.get('peltola_only_explicitly_ranked',0),'Rule-based usable finalist':flags['transfer_to_peltola']},
 {'Destination':'Dan S. Sullivan','First visible named finalist':visible.get('senator_above_peltola_both_ranked',0)+visible.get('senator_only_explicitly_ranked',0),'Rule-based usable finalist':flags['transfer_to_senator']},
 {'Destination':'Neither / inactive','First visible named finalist':visible.get('neither_explicitly_ranked',0),'Rule-based usable finalist':flags['transfer_to_neither_finalist']}
]),'Same 3,940 ballots; two explicitly different measures')
assert descriptive_verification['article_comparison_matches']==descriptive_verification['article_comparison_numeric_cells']==9
article=pd.read_csv(ROOT/'descriptive/visible_finalist_article_comparison.csv')
show.table(article[['governor_first_choice_group','visible_named_finalist','observed_rachel','article_reference','strict_prefix_count']],'Archived November 12 article / November 11 CVR: nine subgroup cells, visible-name definition')


## 3 · Sankey: how Senate ballots transfer

This diagram is rebuilt directly from Rachel’s five Senate rank columns. It follows the predeclared comparison sequence **write-in removed → Daniel J. removed → Heikes removed**. Each ballot goes to its first usable continuing candidate, or to inactive if none remains. Every node and link is checked for conservation, and each stage totals **244,705 ballots**.

Initial allocation uses the first usable choice, so Daniel begins at **3,992**, rather than the literal 3,940 cohort. After receiving write-in transfers his node is **4,055**. The width of each ribbon represents ballots; horizontal placement separates counting stages.

In [ ]:
import build_flows,validate_all_contest_benchmarks
with contextlib.redirect_stdout(io.StringIO()):
    flow=build_flows.build()
    benchmark_validation=validate_all_contest_benchmarks.main()
assert benchmark_validation['matches']==benchmark_validation['cells']==54
assert flow['mass_conservation_passed']
assert flow['benchmark']['matches']==flow['benchmark']['numeric_cells']==18
display(Image(filename=str(ROOT/'flows/senate_sankey.png'),width=1250))
show.table(pd.read_csv(ROOT/'flows/senate_flow_nodes.csv'),'Exact Rachel-only Sankey node counts')


There are **1,363 initially inactive** and **5,549 finally inactive** ballots; **4,186** become newly inactive. Final active counts are **119,933 Peltola** and **119,223 Dan S. Sullivan**. All **18 published Senate round-table numeric values** match the frozen FairVote benchmark after aligning its incremental inactive baseline.

This is the preliminary November 11 snapshot, not a final certified tally or a prediction for ballots absent from these files. The diagram tracks counting allocations, not voters changing their minds.

## 4 · Single-axis MDS: real and randomized consistency

The original method places the four Senate candidates on one axis and checks whether each usable ranking can arise from increasing distance along that axis. We fitted the observed profile and **49 fresh randomized profiles**, preserving first choice and prefix length. Detailed formulas and fitting settings are available below.

## 6 · Original one-axis MDS · fresh fit and fresh randomized profiles

The only model is the original one-dimensional **candidate-axis** method, using the same mathematical core and fitting budget as before. No earlier profiles, positions or simulation results are used as inputs.

**Rachel-only cohort preparation.** Read the Senate CSV directly. Cross a single internal `skipped` token; stop at an `overvote` token or the second consecutive internal skip; retain the first occurrence of repeated candidates. Preserve the occupied rank of a generic write-in before excluding it from the four-candidate model. Empty prefixes remain counted separately. Original Rachel strings remain preserved in the input and linked rows.

**Axis construction.** Let Aᵢⱼ count ballots where j immediately follows i, and Tᵢⱼ count ballots ranking both. Use the inherited symmetric frequency Sᵢⱼ = ½(Aᵢⱼ/Tᵢⱼ + Aⱼᵢ/Tⱼᵢ), with dissimilarity Dᵢⱼ = 1/√Sᵢⱼ. The inherited zero-frequency fallback is 2 divided by the smallest positive S. Fit one-dimensional nonmetric MDS using **20 outer runs × 8 starts**, 1,000-iteration limit and tolerance 10⁻⁶. Align reflected orders, select the modal order, average its coordinates and normalize to [0,1].

**Consistency.** A prefix is consistent if some positive-width interval of voter positions produces its strictly increasing distances to the explicitly ranked candidates. Unranked candidates impose no constraint. One- and two-name prefixes pass automatically; the three-/four-name group is reported separately.

**Randomized comparison.** Generate **49 new profiles**, exactly preserving first named choice × usable-prefix length counts while randomizing distinct ordered tails. Fit and score each randomized profile using its own full-profile axis and the same budget as the observed profile. These are full-profile consistency measures, not held-out predictive accuracy.

In [ ]:
import run_mds,check_mds
from og_axis import summarize,matched_null,seed_for
# This Rachel-only notebook actually reruns the observed fit and every null fit.
with contextlib.redirect_stdout(io.StringIO()):
    mds=run_mds.main()
    check_mds.main()
verification=json.loads((ROOT/'mds/mds_verification.json').read_text())
assert mds['status']=='complete' and len(mds['null'])==49
assert mds['source_sha256']==provenance['inputs']['us_senate']['sha256']
assert verification['status']=='passed' and verification['axes_checked']==51
show.cards()
show.mds_plots()
show.consistency_table()


The fresh Rachel-only results are **61.54% observed consistency versus 41.57% random** for **45,743 three-/four-name prefixes**, a **19.97-point difference**. Across all **243,006 nonempty prefixes**, consistency is **92.76% versus 89.00%**. There are **1,699 empty named prefixes**, retained separately rather than forced into a consistency denominator.

The candidate order is **Peltola → Daniel J. Sullivan Jr. → Heikes → Dan S. Sullivan**, up to reflection. This summarizes ranking structure, not a calibrated ideology scale or evidence of voter motives. The stronger observed-only 100×16 fitting-budget check produces the same consistency; it does not replace the matched 20×8 real-versus-random comparison. Independent strict-distance checks cover **3,264 rankings across all 51 axes**.

The random mean is an estimate from **49 simulations**. Its **Monte Carlo standard error is about 0.62 percentage points** for the three-/four-name group; reporting 41.57% identifies this run’s mean, rather than that degree of population precision.

In [ ]:
show.table(pd.read_csv(ROOT/'mds/candidate_axis.csv').round({'axis_position':6}),'Observed positions from Rachel’s Senate CSV')
show.table(pd.DataFrame([
 {'Check':'MDS inputs','Result':mds['source_filename']},
 {'Check':'Fresh randomized profiles, each with a fresh fit','Result':len(mds['null'])},
 {'Check':'Independent feasibility checks passed','Result':verification['independent_feasibility_rankings_checked']},
 {'Check':'Stronger observed fit: change in 3–4-name consistency','Result':f"{verification['change_three_plus_percentage_points']:.6f} pp"}
]),'Model verification')


## 7 · Source policy, reproducibility and limits

- **Sole ballot sources:** the three original Rachel/FairVote November 11 CSVs included unchanged under `inputs/`. SHA-256 hashes are recorded for each. No older ballot source is read, including to restore hidden overvote candidate identities.
- **Preservation:** all source columns, IDs, rank tokens, first-rank ambiguity annotations and first-rank overvote candidate lists remain in the linked records. A missing governor row remains missing. No irregular ballot is discarded.
- **Reference annotations:** candidate-party labels and stable internal candidate numbers are explicit analysis annotations, not fields supplied in Rachel’s CSVs and not voter registration. The published FairVote HTML is a validation reference only; it does not supply source counts.
- **Primary interpretation:** Alaska-countable preference prefixes are used for headline ranking comparisons, Sankey and MDS. Literal first-choice cohorts are explicitly retained/labeled; the separate first-usable-choice cohort is a sensitivity check. Visible-name comparisons and hidden-identity bounds are supplementary, not headline results.
- **Reproduction:** Run All rebuilds the linked inputs from the three CSVs, recomputes the descriptive tables and Sankey, and freshly fits the observed and all 49 randomized profiles. This run is recorded in `notebook_execution.json`. Historical versions remain preserved outside this active package.

The future-use preference is saved in `ALASKA_2026_ANALYSIS_SOURCE_POLICY.md`: Rachel’s supplied files are the exclusive ballot source for subsequent Alaska 2026 analyses unless you explicitly change that instruction.

In [ ]:
expected_files={x['filename'] for x in provenance['inputs'].values()}
assert RACHEL_INPUT_READS==expected_files,(RACHEL_INPUT_READS,expected_files)
source_audit={'allowed_ballot_input_files':sorted(expected_files),'observed_ballot_input_reads':sorted(RACHEL_INPUT_READS),'old_workspace_ballot_sources_blocked':True,'candidate_parties_are_reference_annotations':True,'all_analyses_freshly_recomputed':True,'alaska_prefix_verifier_executed':usable_verification['status']=='passed','all54_reference_cells_checked_in_Run_All':benchmark_validation['matches']==54,'alaska_prefix_verifier_executed':usable_verification['status']=='passed','all54_reference_cells_checked_in_Run_All':benchmark_validation['matches']==54}
(ROOT/'source_policy_audit.json').write_text(json.dumps(source_audit,indent=2))
show.table(pd.DataFrame([{'Rachel source':slug,'SHA-256':meta['sha256']} for slug,meta in provenance['inputs'].items()]),'Source identity')
print('All Rachel-only notebook checks passed. No older ballot source was accessed.')


## Supplemental validation · all three Rachel contests

Run All reconstructs four stages directly from each Rachel CSV and compares them with the **frozen FairVote-linked RCVis pages explicitly labeled preliminary November 11**. All **54 available numeric candidate/inactive cells match**, with zero mismatches. Every source ballot is retained at each stage; matching the published newly-inactive series requires subtracting the initial inactive baseline only for that reference comparison.

These archived benchmarks are different from the subsequently updated live article and from certified official results. Article comparisons refer specifically to the archived article updated **November 12, 5:30 a.m. Alaska time**, using the **November 11, 6:34 p.m. CVR**. Its nine subgroup cells are the separate supplementary visible-name comparison, not legal transfer counts. The article HTML, fetch record, reference hashes, source hashes, and all 54 comparisons are included in the package.

A separate archived official first-choice summary printed **November 11 at 5:47:54 p.m.** reports **251,088 Senate/House ballots and 250,958 governor ballots**, versus Rachel’s **244,705 and 244,575** records: **6,383 fewer records in each contest**. This is a coverage comparison, not equality to official totals. DOE states its CVR excludes hand-count-only ballots; the aggregate gap alone cannot identify the exact source of every missing record. No absent votes are imputed. This comparison is reproducible with `python3 flows/validate_official_scope.py` against the preserved report.

Alaska’s overvote/skip interpretation follows the [Division of Elections’ 2026 guidance](https://www.elections.alaska.gov/election-results/e/?id=26genr). Matching these preliminary external tables does not independently verify ballot-image adjudication or establish that these CSVs cover the final certified election.